In [2]:
import pandas as pd
import os.path as op
import numpy as np
from stress_risk.utils.data import get_data
#from stress_risk.behavior.utils import get_data
from tqdm.contrib.itertools import product
import matplotlib.pyplot as plt
import pingouin
import seaborn as sns

bids_folder_decod = '/Volumes/mrenkeED/data/ds-stressrisk'
bids_folder_behav ='/Users/mrenke/data/ds-stressrisk'

bids_folder_decod = bids_folder_behav

In [3]:
df = get_data(bids_folder = bids_folder_behav)

In [3]:
from utils import get_decoding_info

subjects = df.reset_index()['subject'].unique()
sessions = [1, 2]

pred = []
for (sub, session) in product(subjects, sessions):
    #print(sub, session)
    pred.append(get_decoding_info(sub, session,bids_folder=bids_folder_decod))

100%|██████████| 100/100 [00:02<00:00, 39.15it/s]


In [4]:
pred = pd.concat(pred)
pred

E        sd
subject session mask  n_voxels trial_nr                    
1       1       NPC_R 250      1         2.274240  0.004388
                               2         2.653213  0.000303
                               3         2.564742  0.000500
                               4         2.609074  0.008393
                               5         2.508921  0.002820
...                                           ...       ...
61      2       NPC_R 250      116       3.000265  0.003114
                               117       3.074664  0.000342
                               118       2.708719  0.009328
                               119       2.342731  0.002574
                               120       1.913491  0.001156

[12000 rows x 2 columns]

In [5]:
pred = pred.join(df, how='inner')
pred

E  session        rt  \
subject trial_nr session mask  n_voxels run                                
1       1        1       NPC_R 250      1    2.274240        1  0.476974   
                                        1    2.274240        2  1.843622   
                 2       NPC_R 250      1    3.208038        1  0.476974   
                                        1    3.208038        2  1.843622   
        2        1       NPC_R 250      1    2.653213        1  0.505059   
...                                               ...      ...       ...   
61      119      2       NPC_R 250      6    2.342731        2  1.873285   
        120      1       NPC_R 250      6    3.000113        1  2.106819   
                                        6    3.000113        2  0.989251   
                 2       NPC_R 250      6    1.913491        1  2.106819   
                                        6    1.913491        2  0.989251   

                                               n1    n2  prob1  prob2  choice  \
subject trial_nr session mask  n_voxels run                                     
1       1        1       NPC_R 250      1    13.0  10.0   0.55   1.00    True   
                                        1    28.0  32.0   1.00   0.55   False   
                 2       NPC_R 250      1    13.0  10.0   0.55   1.00    True   
                                        1    28.0  32.0   1.00   0.55   False   
        2        1       NPC_R 250      1     8.0   7.0   0.55   1.00    True   
...                                           ...   ...    ...    ...     ...   
61      119      2       NPC_R 250      6    34.0  20.0   0.55   1.00    True   
        120      1       NPC_R 250      6     7.0  15.0   1.00   0.55    True   
                                        6    42.0  28.0   0.55   1.00    True   
                 2       NPC_R 250      6     7.0  15.0   1.00   0.55    True   
                                        6    42.0  28.0   0.55   1.00    True   

                                             risky_first  chose_risky  \
subject trial_nr session mask  n_voxels run                             
1       1        1       NPC_R 250      1           True        False   
                                        1          False        False   
                 2       NPC_R 250      1           True        False   
                                        1          False        False   
        2        1       NPC_R 250      1           True        False   
...                                                  ...          ...   
61      119      2       NPC_R 250      6           True        False   
        120      1       NPC_R 250      6          False         True   
                                        6           True        False   
                 2       NPC_R 250      6          False         True   
                                        6           True        False   

                                             n_risky  n_safe      frac  \
subject trial_nr session mask  n_voxels run                              
1       1        1       NPC_R 250      1       13.0    10.0  1.300000   
                                        1       32.0    28.0  1.142857   
                 2       NPC_R 250      1       13.0    10.0  1.300000   
                                        1       32.0    28.0  1.142857   
        2        1       NPC_R 250      1        8.0     7.0  1.142857   
...                                              ...     ...       ...   
61      119      2       NPC_R 250      6       34.0    20.0  1.700000   
        120      1       NPC_R 250      6       15.0     7.0  2.142857   
                                        6       42.0    28.0  1.500000   
                 2       NPC_R 250      6       15.0     7.0  2.142857   
                                        6       42.0    28.0  1.500000   

                                             log(risky/safe)   log(n1)  \
subject trial_nr sessi

In [6]:
p = pred 


In [7]:
pp = pd.DataFrame(data={'E': pred['E'], 'log(n1)' :pred['log(n1)']})
pp.reset_index().set_index(['subject','session'])
pp.groupby(['subject','session'])

In [8]:
r2_ = pp.groupby(['subject','session']).apply(lambda d: pingouin.corr(d['E'], d['log(n1)']))
r2 = r2_.groupby(['subject','session']).mean()
r_final = r2[['r']]
r_final


r
subject session          
1       1        0.061686
        2        0.104811
2       1        0.013186
        2       -0.024392
3       1        0.037224
...                   ...
58      2        0.170520
59      1        0.313141
        2        0.145500
61      1        0.144704
        2        0.106332

[100 rows x 1 columns]

In [11]:
r_final.to_csv('~/data/ds-stressrisk/derivatives/decoding_accuracy_allSub_allSes.csv')
r_final.to_csv('/Users/mrenke/git/stress_risk/stress_risk/analyse_results/decoding_accuracy_allSub_allSes.csv')

In [6]:
pred.groupby(['subject','session'])['sd'].mean().to_csv('~/data/ds-stressrisk/derivatives/decoding_sd_allSub_allSes.csv')